# Proyecto de Minería de Datos
## Modelo de Regresión Lineal

### Proyecto: Predicción del porcentaje de apoyo al equipo local

En este notebook construiremos un **modelo de Regresión Lineal** para responder:

> **¿Qué porcentaje de personas apoyará al equipo local antes de un partido?**

**Variable objetivo:** `Bet_Perc_on_Home`

Esta variable es numérica, por eso se puede utilizar regresión.

### Diferencia con el trabajo anterior
En el Árbol de Decisión de Regresión también predecíamos un número. Aquí usaremos una **Regresión Lineal**, que intenta representar la relación entre las variables mediante una ecuación:

\[
Y = b_0 + b_1X_1 + b_2X_2 + ... + b_nX_n
\]

- `Y`: porcentaje de apoyo al local.
- `X`: estadísticas de los equipos.
- `b`: coeficientes del modelo.


# 1. Definición del problema y variable objetivo

Queremos estimar `Bet_Perc_on_Home`, es decir, el porcentaje de apoyo hacia el equipo local.

### ¿Por qué esta variable?
Porque es una variable **continua y numérica**, requisito para aplicar Regresión Lineal.

### Variables independientes
Usaremos estadísticas previas al partido como:
- posición en liga,
- puntos,
- partidos jugados,
- victorias, empates y derrotas,
- goles anotados y recibidos,
- diferencia de goles,
- porcentajes históricos,
- enfrentamientos previos.

### Evitar fuga de información
No usaremos `Votes_for_Home`, `Votes_for_Draw`, `Votes_for_Away` ni `Total_Bettors`, porque podrían estar directamente relacionados con el porcentaje que queremos predecir.

### Para exponer
> “Buscamos estimar el porcentaje de apoyo al equipo local utilizando estadísticas previas de ambos equipos, evitando variables que puedan revelar directamente la respuesta.”


# 2. Importación de librerías

- `pandas`: datos.
- `numpy`: cálculos.
- `matplotlib`: gráficos.
- `sklearn`: entrenamiento y evaluación.
- `statsmodels`: coeficientes y p-valores.

Si `statsmodels` no está instalado, ejecuta en tu terminal:

```bash
python -m pip install statsmodels
```


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

import statsmodels.api as sm

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Librerías cargadas correctamente.")


# 3. Carga del dataset

El notebook debe estar dentro de `Codigo/` y el dataset en:

`Data/raw/predicciones_personas.csv`

Por eso utilizamos la ruta relativa:

`../Data/raw/predicciones_personas.csv`


In [ ]:
ruta = "../Data/raw/predicciones_personas.csv"
datos = pd.read_csv(ruta)

print("Dataset cargado correctamente.")
print("Dimensiones:", datos.shape)
display(datos.head())


## ¿Qué significa `datos.shape`?

Tiene la forma `(filas, columnas)`.

### Para exponer
> “Primero cargamos el dataset y verificamos cuántos registros y variables contiene.”


# 4. Revisión de la variable objetivo

Antes de entrenar, comprobamos que `Bet_Perc_on_Home` exista y revisamos sus estadísticas.


In [ ]:
objetivo = "Bet_Perc_on_Home"

if objetivo not in datos.columns:
    raise ValueError(f"No existe la columna {objetivo}")

print("Variable objetivo:", objetivo)
display(datos[objetivo].describe())

print("Nulos en la variable objetivo:")
print(datos[objetivo].isna().sum())


## ¿Cómo leer `describe()`?

- `count`: cantidad de valores.
- `mean`: promedio.
- `std`: dispersión.
- `min`: mínimo.
- `50%`: mediana.
- `max`: máximo.

### Para exponer
> “Revisamos la variable objetivo para comprobar que sea numérica y conocer su comportamiento general.”


# 5. Selección de variables independientes

El modelo utilizará estadísticas deportivas previas al partido. El código toma únicamente las columnas que realmente existen en el dataset.


In [ ]:
numericas_candidatas = [
    "League_Rank_1", "League_Rank_2",
    "Games_played_1", "Games_played_2",
    "Points_1", "Points_2",
    "Won_1", "Won_2",
    "Draw_1", "Draw_2",
    "Lost_1", "Lost_2",
    "Goals_Scored_1", "Goals_Scored_2",
    "Goals_Rec_1", "Goal_Rec_2",
    "Goals_Diff_1", "Goals_Diff_2",
    "Win_Perc_1", "Win_Perc_2",
    "Draw_Perc_1", "Draw_Perc_2",
    "Number_of_H2H_matches"
]

categoricas_candidatas = ["Weekday", "League_type_country"]

numericas = [c for c in numericas_candidatas if c in datos.columns]
categoricas = [c for c in categoricas_candidatas if c in datos.columns]
variables = numericas + categoricas

print("Variables numéricas:")
print(numericas)

print("\nVariables categóricas:")
print(categoricas)

print("\nTotal:", len(variables))


## ¿Por qué estas variables?

Porque describen el rendimiento previo de los equipos. Por ejemplo, una mejor posición, más puntos o más goles podrían estar asociados con mayor apoyo.

Esto representa **asociación**, no causalidad.

### Para exponer
> “Seleccionamos variables relacionadas con el rendimiento porque pueden ayudar a explicar la preferencia de las personas.”


# 6. Limpieza, codificación y estandarización

Haremos:
1. Eliminar filas sin variable objetivo.
2. Completar nulos numéricos con la mediana.
3. Completar nulos categóricos con la moda.
4. Convertir texto a números mediante One-Hot Encoding.
5. Estandarizar variables numéricas.

### ¿Por qué estandarizar?
Porque las variables usan escalas distintas. Al estandarizarlas podemos comparar mejor el tamaño de los coeficientes.


In [ ]:
datos_modelo = datos.dropna(subset=[objetivo]).copy()

X = datos_modelo[variables]
y = datos_modelo[objetivo]

num_pipe = Pipeline([
    ("imputador", SimpleImputer(strategy="median")),
    ("escalador", StandardScaler())
])

transformadores = [
    ("numericas", num_pipe, numericas)
]

if categoricas:
    cat_pipe = Pipeline([
        ("imputador", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            drop="first",
            sparse_output=False
        ))
    ])
    transformadores.append(("categoricas", cat_pipe, categoricas))

preprocesador = ColumnTransformer(transformadores)

print("Registros:", len(datos_modelo))
print("X:", X.shape)
print("y:", y.shape)


## ¿Qué significa `drop="first"`?

En One-Hot Encoding eliminamos una categoría de referencia para evitar redundancia matemática entre las columnas.

### Para exponer
> “Limpiamos valores faltantes, convertimos las categorías a formato numérico y estandarizamos las variables para interpretar mejor sus coeficientes.”


# 7. División en entrenamiento y prueba

- 80% entrenamiento.
- 20% prueba.

El modelo aprende con entrenamiento y se evalúa con datos que no vio antes.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

print("Entrenamiento %:", round(len(X_train)/len(X)*100, 2))
print("Prueba %:", round(len(X_test)/len(X)*100, 2))


### Para exponer
> “Utilizamos el 80% de los datos para entrenar y el 20% para comprobar qué tan bien funciona el modelo con datos nuevos.”


# 8. Entrenamiento de la Regresión Lineal

`LinearRegression` intenta encontrar los coeficientes que mejor relacionan las variables independientes con la variable objetivo.


In [ ]:
modelo_lineal = Pipeline([
    ("preprocesador", preprocesador),
    ("modelo", LinearRegression())
])

modelo_lineal.fit(X_train, y_train)
predicciones = modelo_lineal.predict(X_test)

print("Modelo entrenado correctamente.")


# 9. Evaluación del modelo

### MSE
Error Cuadrático Medio. Penaliza especialmente los errores grandes.

> **Menor MSE = mejor.**

### R²
Indica qué parte de la variación del porcentaje de apoyo logra explicar el modelo.

- Cerca de 1: explica bastante.
- Cerca de 0: explica poco.
- Menor a 0: peor que usar una predicción básica basada en el promedio.

> **Mayor R² = mejor.**

### MAE
Error Absoluto Medio.

Si `MAE = 8`, el modelo se equivoca aproximadamente **8 puntos porcentuales** en promedio.


In [ ]:
mse = mean_squared_error(y_test, predicciones)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, predicciones)
r2 = r2_score(y_test, predicciones)

print("=" * 45)
print("RESULTADOS")
print("=" * 45)
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R²   : {r2:.4f}")


### Para exponer
> “Evaluamos el modelo principalmente con MSE y R². También usamos MAE porque permite expresar el error promedio en puntos porcentuales.”


# 10. Coeficientes del modelo

Cada variable recibe un coeficiente.

### Coeficiente positivo
Al aumentar esa variable, la predicción tiende a aumentar, manteniendo las demás constantes.

### Coeficiente negativo
Al aumentar esa variable, la predicción tiende a disminuir, manteniendo las demás constantes.

Como las variables numéricas están estandarizadas, sus coeficientes son más comparables.


In [ ]:
prep = modelo_lineal.named_steps["preprocesador"]
regresion = modelo_lineal.named_steps["modelo"]

nombres = prep.get_feature_names_out()
coeficientes = regresion.coef_

tabla_coef = pd.DataFrame({
    "Variable": nombres,
    "Coeficiente": coeficientes,
    "Impacto_absoluto": np.abs(coeficientes)
}).sort_values("Impacto_absoluto", ascending=False)

display(tabla_coef.head(20))


## ¿Cómo interpretar los coeficientes?

- Signo positivo: asociación positiva.
- Signo negativo: asociación negativa.
- Mayor valor absoluto: mayor impacto estimado dentro del modelo.

No debemos decir que una variable **causa** el apoyo. Solo hablamos de relación estadística.


# 11. Significancia estadística y p-valores

La actividad pide analizar los **p-valores**.

Regla habitual:

- `p < 0.05` → variable estadísticamente significativa al 5%.
- `p >= 0.05` → no hay suficiente evidencia estadística para considerarla significativa al 5%.

Un p-valor bajo **no demuestra causalidad**.


In [ ]:
# Transformar únicamente los datos de entrenamiento
X_train_t = prep.transform(X_train)
X_train_t = np.asarray(X_train_t, dtype=float)

# Agregar intercepto
X_ols = sm.add_constant(X_train_t, has_constant="add")

# Ajustar OLS
modelo_ols = sm.OLS(
    np.asarray(y_train, dtype=float),
    X_ols
).fit()

print(modelo_ols.summary())


## ¿Qué mirar en el resumen?

- `coef`: coeficiente.
- `P>|t|`: p-valor.
- `R-squared`: R² del análisis estadístico.

### Para exponer
> “Usamos los p-valores para identificar qué variables muestran evidencia estadística de relación con el porcentaje de apoyo.”


In [ ]:
nombres_ols = ["Intercepto"] + list(nombres)

tabla_p = pd.DataFrame({
    "Variable": nombres_ols,
    "Coeficiente": modelo_ols.params,
    "P_valor": modelo_ols.pvalues
})

tabla_p["Significativa_5%"] = np.where(
    tabla_p["P_valor"] < 0.05,
    "Sí",
    "No"
)

tabla_p["Impacto_absoluto"] = np.abs(tabla_p["Coeficiente"])

tabla_p = tabla_p.sort_values(
    "Impacto_absoluto",
    ascending=False
)

display(tabla_p.head(25))


# 12. Gráfico de coeficientes

- Derecha: relación positiva.
- Izquierda: relación negativa.
- Barra más larga: mayor impacto estimado.


In [ ]:
top = tabla_coef.head(15).sort_values("Coeficiente")

plt.figure(figsize=(10, 7))
plt.barh(top["Variable"], top["Coeficiente"])
plt.title("Variables con mayor impacto según coeficientes")
plt.xlabel("Coeficiente")
plt.ylabel("Variable")
plt.tight_layout()
plt.show()


### Para exponer
> “Este gráfico muestra las variables con mayor peso dentro del modelo y si su relación estimada con el apoyo es positiva o negativa.”


# 13. Valores reales vs predichos

La diagonal representa una predicción perfecta.

Mientras más cerca esté un punto de la diagonal, más cercana fue la predicción al valor real.


In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(y_test, predicciones, alpha=0.5)

minimo = min(y_test.min(), predicciones.min())
maximo = max(y_test.max(), predicciones.max())

plt.plot(
    [minimo, maximo],
    [minimo, maximo],
    linestyle="--",
    label="Predicción perfecta"
)

plt.xlabel("Porcentaje real de apoyo")
plt.ylabel("Porcentaje predicho")
plt.title("Valores reales vs predichos")
plt.legend()
plt.tight_layout()
plt.show()


### Para exponer
> “Cada punto representa un partido. Mientras más cerca esté de la diagonal, más cercana fue la predicción al valor real.”


# 14. Análisis de residuos

Un residuo es:

`valor real - valor predicho`

Idealmente, los errores deberían estar alrededor de cero sin un patrón demasiado claro.


In [ ]:
residuos = y_test - predicciones

plt.figure(figsize=(8, 6))
plt.scatter(predicciones, residuos, alpha=0.5)
plt.axhline(y=0, linestyle="--")

plt.xlabel("Valores predichos")
plt.ylabel("Residuos")
plt.title("Residuos del modelo")
plt.tight_layout()
plt.show()


### Para exponer
> “Los residuos muestran cuánto se equivocó el modelo. Buscamos que estén distribuidos alrededor de cero y sin un patrón fuerte.”


# 15. Relación entre una variable independiente y la dependiente

Para cumplir la parte visual de la actividad, seleccionamos automáticamente la variable numérica con mayor coeficiente absoluto y la comparamos con `Bet_Perc_on_Home`.


In [ ]:
impactos_num = []

for variable in numericas:
    indices = [
        i for i, nombre in enumerate(nombres)
        if nombre.endswith(variable)
    ]
    if indices:
        i = indices[0]
        impactos_num.append((variable, abs(coeficientes[i])))

if impactos_num:
    variable_grafico = max(impactos_num, key=lambda x: x[1])[0]

    dg = datos_modelo[
        [variable_grafico, objetivo]
    ].dropna()

    plt.figure(figsize=(8, 6))
    plt.scatter(
        dg[variable_grafico],
        dg[objetivo],
        alpha=0.4
    )

    plt.xlabel(variable_grafico)
    plt.ylabel("Porcentaje de apoyo al local")
    plt.title(
        f"Relación entre {variable_grafico} "
        "y el porcentaje de apoyo"
    )
    plt.tight_layout()
    plt.show()

    print("Variable seleccionada:", variable_grafico)
else:
    print("No se encontró variable numérica para graficar.")


### Para exponer
> “Este gráfico permite observar visualmente cómo se relaciona una de las variables más importantes con el porcentaje de apoyo al equipo local.”


# 16. Variables estadísticamente significativas

Aquí filtramos automáticamente las variables con p-valor menor a 0.05.


In [ ]:
significativas = tabla_p[
    (tabla_p["P_valor"] < 0.05) &
    (tabla_p["Variable"] != "Intercepto")
].copy()

significativas = significativas.sort_values(
    "Impacto_absoluto",
    ascending=False
)

print("Cantidad de variables significativas al 5%:")
print(len(significativas))

display(significativas.head(10))


# 17. Conclusiones automáticas

Esta celda genera conclusiones usando **los resultados reales del modelo**, para no inventar porcentajes.


In [ ]:
print("CONCLUSIONES")
print("-" * 65)

print(
    f"• La Regresión Lineal obtuvo un MSE de {mse:.2f}."
)

print(
    f"• El modelo obtuvo un R² de {r2:.3f}."
)

print(
    f"• El error promedio fue de aproximadamente "
    f"{mae:.2f} puntos porcentuales."
)

if r2 > 0:
    print(
        "• El modelo logra explicar parte de la variación "
        "del porcentaje de apoyo."
    )
else:
    print(
        "• El modelo no logra explicar adecuadamente "
        "la variación del porcentaje de apoyo."
    )

if len(significativas) > 0:
    principal = significativas.iloc[0]

    sentido = (
        "positiva"
        if principal["Coeficiente"] > 0
        else "negativa"
    )

    print(
        f"• Entre las variables significativas, "
        f"{principal['Variable']} presentó uno de los "
        f"mayores impactos."
    )

    print(
        f"• Su relación estimada fue {sentido} "
        f"y su p-valor fue {principal['P_valor']:.4f}."
    )
else:
    print(
        "• No se encontraron variables con p-valor "
        "menor a 0.05."
    )

print(
    "• Los coeficientes permiten identificar la dirección "
    "y el peso de las relaciones estimadas."
)


# 18. Conclusiones para la exposición

Después de ejecutar el notebook, resume en puntos:

- El modelo predice el **porcentaje de apoyo al equipo local**.
- El **MSE** mide el error cuadrático: menor es mejor.
- El **R²** indica cuánto logra explicar el modelo: mayor es mejor.
- El **MAE** indica aproximadamente cuántos puntos porcentuales se equivoca.
- Los **coeficientes** muestran la dirección e intensidad de la relación.
- Los **p-valores** permiten identificar variables estadísticamente significativas.
- Los gráficos muestran qué tan cerca están las predicciones de los valores reales.

### Frase corta para cerrar
> “La Regresión Lineal nos permitió estimar el porcentaje de apoyo al equipo local y analizar qué variables presentan mayor relación estadística con esa predicción.”

**Usa siempre los valores generados por el código. No inventes resultados.**


# 19. Resumen rápido para estudiar

### ¿Qué predecimos?
El porcentaje de apoyo al equipo local.

### ¿Qué tipo de problema es?
Regresión.

### ¿Qué modelo usamos?
`LinearRegression`.

### ¿División de datos?
80% entrenamiento y 20% prueba.

### ¿Nulos?
Mediana para números y moda para categorías.

### ¿Variables categóricas?
One-Hot Encoding.

### ¿Por qué estandarizamos?
Para comparar mejor los coeficientes.

### ¿MSE?
Error cuadrático medio. Menor es mejor.

### ¿R²?
Qué tanto de la variación explica el modelo. Mayor es mejor.

### ¿MAE?
Error promedio en puntos porcentuales.

### ¿Coeficiente positivo?
La predicción tiende a aumentar cuando aumenta esa variable, manteniendo las demás constantes.

### ¿Coeficiente negativo?
La predicción tiende a disminuir.

### ¿p-valor?
Sirve para evaluar significancia estadística.

### ¿Cuándo una variable es significativa?
Habitualmente cuando `p < 0.05`.

### ¿Un p-valor bajo demuestra causalidad?
No.

### ¿Residuo?
Diferencia entre el valor real y el predicho.
